<a href="https://colab.research.google.com/github/tiendat1806-06/TRITUENHANTAO/blob/main/BTvenhaTuan3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import time

WIN_K = {3: 3, 5: 5, 10: 5}

def get_k(n):
    return WIN_K.get(n, 5)

def make_board(n):
    return [[0] * n for _ in range(n)]

def print_board(board):
    n = len(board)
    sym = {0: ".", 1: "X", -1: "O"}
    col_idx = "   " + "  ".join(str(c).rjust(2) for c in range(n))
    print(col_idx)
    for r in range(n):
        row_str = f"{r:2} " + "  ".join(f" {sym[board[r][c]]}" for c in range(n))
        print(row_str)
    print()

def get_moves(board):
    n = len(board)
    return [(r, c) for r in range(n) for c in range(n) if board[r][c] == 0]

def apply_move(board, r, c, player):
    board[r][c] = player

def undo_move(board, r, c):
    board[r][c] = 0

def check_winner(board, r, c, player, k):
    n = len(board)
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
    for dr, dc in directions:
        count = 1
        for sign in [1, -1]:
            nr, nc = r + sign * dr, c + sign * dc
            while 0 <= nr < n and 0 <= nc < n and board[nr][nc] == player:
                count += 1
                nr += sign * dr
                nc += sign * dc
        if count >= k:
            return True
    return False

def is_full(board):
    return all(board[r][c] != 0 for r in range(len(board)) for c in range(len(board[0])))

def score_line(window, k, player):
    opp = -player
    p_count = window.count(player)
    o_count = window.count(opp)
    if o_count > 0:
        return 0
    score_map = {1: 1, 2: 10, 3: 100, 4: 1000}
    if p_count == k:
        return 100000
    return score_map.get(p_count, 0)

def heuristic(board, player, k):
    n = len(board)
    score = 0
    opp = -player

    def eval_window(window):
        return score_line(window, k, player) - score_line(window, k, opp)

    for r in range(n):
        for c in range(n - k + 1):
            score += eval_window([board[r][c + i] for i in range(k)])
    for r in range(n - k + 1):
        for c in range(n):
            score += eval_window([board[r + i][c] for i in range(k)])
    for r in range(n - k + 1):
        for c in range(n - k + 1):
            score += eval_window([board[r + i][c + i] for i in range(k)])
    for r in range(k - 1, n):
        for c in range(n - k + 1):
            score += eval_window([board[r - i][c + i] for i in range(k)])
    return score

def order_moves(board, moves):
    n = len(board)
    center = n / 2
    return sorted(moves, key=lambda m: abs(m[0] - center) + abs(m[1] - center))

def minimax(board, depth, alpha, beta, is_max, last_move, k, max_depth):
    if last_move:
        r, c = last_move
        player_just_moved = 1 if not is_max else -1
        if check_winner(board, r, c, player_just_moved, k):
            return 100000 + depth if player_just_moved == 1 else -(100000 + depth), None

    moves = get_moves(board)
    if not moves:
        return 0, None
    if depth == 0:
        return heuristic(board, 1, k), None

    moves = order_moves(board, moves)
    best_move = moves[0]

    if is_max:
        best = -math.inf
        for r, c in moves:
            apply_move(board, r, c, 1)
            val, _ = minimax(board, depth - 1, alpha, beta, False, (r, c), k, max_depth)
            undo_move(board, r, c)
            if val > best:
                best, best_move = val, (r, c)
            alpha = max(alpha, best)
            if beta <= alpha:
                break
        return best, best_move
    else:
        best = math.inf
        for r, c in moves:
            apply_move(board, r, c, -1)
            val, _ = minimax(board, depth - 1, alpha, beta, True, (r, c), k, max_depth)
            undo_move(board, r, c)
            if val < best:
                best, best_move = val, (r, c)
            beta = min(beta, best)
            if beta <= alpha:
                break
        return best, best_move

def get_max_depth(n):
    if n <= 3:
        return 9
    elif n <= 5:
        return 4
    elif n <= 10:
        return 3
    else:
        return 2

def play_game(n):
    board = make_board(n)
    k = get_k(n)
    max_depth = get_max_depth(n)
    print(f"\n=== Tic-Tac-Toe {n}x{n} | Thắng khi có {k} quân liên tiếp ===")
    print(f"Bạn: X (player 1) | AI: O (player -1) | Depth tìm kiếm: {max_depth}\n")

    current = 1
    last_move = None

    while True:
        print_board(board)
        moves = get_moves(board)
        if not moves:
            print("Hòa!")
            break

        if current == 1:
            print("Lượt của bạn (X). Nhập hàng và cột (vd: 2 3): ", end="")
            try:
                r, c = map(int, input().split())
            except ValueError:
                print("Nhập không hợp lệ!")
                continue
            if (r, c) not in moves:
                print("Ô đã có quân hoặc ngoài bảng!")
                continue
        else:
            print("AI đang suy nghĩ...")
            t0 = time.time()
            _, move = minimax(board, max_depth, -math.inf, math.inf, False, last_move, k, max_depth)
            r, c = move
            print(f"AI đặt tại ({r}, {c}) | Thời gian: {time.time()-t0:.2f}s")

        apply_move(board, r, c, current)
        last_move = (r, c)

        if check_winner(board, r, c, current, k):
            print_board(board)
            winner = "Bạn (X)" if current == 1 else "AI (O)"
            print(f"🎉 {winner} thắng!")
            break

        current = -current

def main():

    print(" MINIMAX TIC-TAC-TOE (NxN)   ")

    print("Chọn kích thước bảng:")
    print("  1. 3x3  (K=3, classic)")
    print("  2. 5x5  (K=5)")
    print("  3. 10x10 (K=5, gomoku)")
    print("  4. NxN  (tùy chỉnh)")
    choice = input("Lựa chọn (1-4): ").strip()

    size_map = {"1": 3, "2": 5, "3": 10}
    if choice in size_map:
        n = size_map[choice]
    elif choice == "4":
        n = int(input("Nhập N: "))
    else:
        print("Lựa chọn không hợp lệ.")
        return

    play_game(n)

if __name__ == "__main__":
    main()

 MINIMAX TIC-TAC-TOE (NxN)   
Chọn kích thước bảng:
  1. 3x3  (K=3, classic)
  2. 5x5  (K=5)
  3. 10x10 (K=5, gomoku)
  4. NxN  (tùy chỉnh)
Lựa chọn (1-4): 2

=== Tic-Tac-Toe 5x5 | Thắng khi có 5 quân liên tiếp ===
Bạn: X (player 1) | AI: O (player -1) | Depth tìm kiếm: 4

    0   1   2   3   4
 0  .   .   .   .   .
 1  .   .   .   .   .
 2  .   .   .   .   .
 3  .   .   .   .   .
 4  .   .   .   .   .

Lượt của bạn (X). Nhập hàng và cột (vd: 2 3): 1 0
    0   1   2   3   4
 0  .   .   .   .   .
 1  X   .   .   .   .
 2  .   .   .   .   .
 3  .   .   .   .   .
 4  .   .   .   .   .

AI đang suy nghĩ...
AI đặt tại (2, 2) | Thời gian: 0.09s
    0   1   2   3   4
 0  .   .   .   .   .
 1  X   .   .   .   .
 2  .   .   O   .   .
 3  .   .   .   .   .
 4  .   .   .   .   .

Lượt của bạn (X). Nhập hàng và cột (vd: 2 3): 0 0
    0   1   2   3   4
 0  X   .   .   .   .
 1  X   .   .   .   .
 2  .   .   O   .   .
 3  .   .   .   .   .
 4  .   .   .   .   .

AI đang suy nghĩ...
AI đặt tại (3, 0)

In [ ]:
import math
import time

WIN_K = {3: 3, 5: 5, 10: 5}
pruned_count = 0

def get_k(n):
    return WIN_K.get(n, 5)

def make_board(n):
    return [[0] * n for _ in range(n)]

def print_board(board):
    n = len(board)
    sym = {0: ".", 1: "X", -1: "O"}
    print("   " + "  ".join(str(c).rjust(2) for c in range(n)))
    for r in range(n):
        print(f"{r:2} " + "  ".join(f" {sym[board[r][c]]}" for c in range(n)))
    print()

def get_moves(board):
    n = len(board)
    return [(r, c) for r in range(n) for c in range(n) if board[r][c] == 0]

def apply_move(board, r, c, player):
    board[r][c] = player

def undo_move(board, r, c):
    board[r][c] = 0

def check_winner(board, r, c, player, k):
    n = len(board)
    for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
        count = 1
        for sign in [1, -1]:
            nr, nc = r + sign * dr, c + sign * dc
            while 0 <= nr < n and 0 <= nc < n and board[nr][nc] == player:
                count += 1
                nr += sign * dr
                nc += sign * dc
        if count >= k:
            return True
    return False

def score_line(window, k, player):
    if window.count(-player) > 0:
        return 0
    p = window.count(player)
    if p == k:
        return 100000
    return {1: 1, 2: 10, 3: 100, 4: 1000}.get(p, 0)

def heuristic(board, k):
    n = len(board)
    score = 0

    def eval_window(w):
        return score_line(w, k, 1) - score_line(w, k, -1)

    for r in range(n):
        for c in range(n - k + 1):
            score += eval_window([board[r][c + i] for i in range(k)])
    for r in range(n - k + 1):
        for c in range(n):
            score += eval_window([board[r + i][c] for i in range(k)])
    for r in range(n - k + 1):
        for c in range(n - k + 1):
            score += eval_window([board[r + i][c + i] for i in range(k)])
    for r in range(k - 1, n):
        for c in range(n - k + 1):
            score += eval_window([board[r - i][c + i] for i in range(k)])
    return score

def order_moves(board, moves):
    center = len(board) / 2
    return sorted(moves, key=lambda m: abs(m[0] - center) + abs(m[1] - center))

def alpha_beta(board, depth, alpha, beta, is_max, last_move, k):
    global pruned_count

    if last_move:
        r, c = last_move
        mover = 1 if not is_max else -1
        if check_winner(board, r, c, mover, k):
            return (100000 + depth) * mover, None

    moves = get_moves(board)
    if not moves:
        return 0, None
    if depth == 0:
        return heuristic(board, k), None

    moves = order_moves(board, moves)
    best_move = moves[0]

    if is_max:
        best = -math.inf
        for r, c in moves:
            apply_move(board, r, c, 1)
            val, _ = alpha_beta(board, depth - 1, alpha, beta, False, (r, c), k)
            undo_move(board, r, c)
            if val > best:
                best, best_move = val, (r, c)
            alpha = max(alpha, best)
            if beta <= alpha:
                pruned_count += 1
                break
        return best, best_move
    else:
        best = math.inf
        for r, c in moves:
            apply_move(board, r, c, -1)
            val, _ = alpha_beta(board, depth - 1, alpha, beta, True, (r, c), k)
            undo_move(board, r, c)
            if val < best:
                best, best_move = val, (r, c)
            beta = min(beta, best)
            if beta <= alpha:
                pruned_count += 1
                break
        return best, best_move

def get_depth(n):
    if n <= 3:  return 9
    elif n <= 5: return 4
    elif n <= 10: return 3
    else:        return 2

def play_game(n):
    global pruned_count
    board = make_board(n)
    k = get_k(n)
    depth = get_depth(n)
    print(f"\n=== Tic-Tac-Toe {n}x{n} | K={k} | Alpha-Beta Pruning | Depth={depth} ===")
    print("Bạn: X  |  AI: O\n")

    current = 1
    last_move = None

    while True:
        print_board(board)
        moves = get_moves(board)
        if not moves:
            print("Hòa!")
            break

        if current == 1:
            print("Lượt của bạn (X). Nhập hàng cột (vd: 2 3): ", end="")
            try:
                r, c = map(int, input().split())
            except ValueError:
                print("Nhập không hợp lệ!")
                continue
            if (r, c) not in moves:
                print("Ô không hợp lệ!")
                continue
        else:
            pruned_count = 0
            t0 = time.time()
            _, move = alpha_beta(board, depth, -math.inf, math.inf, False, last_move, k)
            elapsed = time.time() - t0
            r, c = move
            print(f"AI đặt ({r},{c}) | Thời gian: {elapsed:.3f}s | Nhánh bị cắt: {pruned_count}")

        apply_move(board, r, c, current)
        last_move = (r, c)

        if check_winner(board, r, c, current, k):
            print_board(board)
            print(f"{'Bạn (X)' if current == 1 else 'AI (O)'} thắng!")
            break

        current = -current

def main():

    print("  ALPHA-BETA PRUNING TIC-TAC-TOE NxN  ")

    print("1. 3x3   (K=3)")
    print("2. 5x5   (K=5)")
    print("3. 10x10 (K=5)")
    print("4. NxN   (tùy chỉnh)")
    choice = input("Lựa chọn: ").strip()

    size_map = {"1": 3, "2": 5, "3": 10}
    if choice in size_map:
        n = size_map[choice]
    elif choice == "4":
        n = int(input("Nhập N: "))
    else:
        print("Không hợp lệ.")
        return

    play_game(n)

if __name__ == "__main__":
    main()

  ALPHA-BETA PRUNING TIC-TAC-TOE NxN  
1. 3x3   (K=3)
2. 5x5   (K=5)
3. 10x10 (K=5)
4. NxN   (tùy chỉnh)
Lựa chọn: 1

=== Tic-Tac-Toe 3x3 | K=3 | Alpha-Beta Pruning | Depth=9 ===
Bạn: X  |  AI: O

    0   1   2
 0  .   .   .
 1  .   .   .
 2  .   .   .

Lượt của bạn (X). Nhập hàng cột (vd: 2 3): 0 0
    0   1   2
 0  X   .   .
 1  .   .   .
 2  .   .   .

AI đặt (1,1) | Thời gian: 0.020s | Nhánh bị cắt: 1163
    0   1   2
 0  X   .   .
 1  .   O   .
 2  .   .   .

Lượt của bạn (X). Nhập hàng cột (vd: 2 3): 1 0
    0   1   2
 0  X   .   .
 1  X   O   .
 2  .   .   .

AI đặt (2,0) | Thời gian: 0.002s | Nhánh bị cắt: 159
    0   1   2
 0  X   .   .
 1  X   O   .
 2  O   .   .

Lượt của bạn (X). Nhập hàng cột (vd: 2 3): 0 2
    0   1   2
 0  X   .   X
 1  X   O   .
 2  O   .   .

AI đặt (0,1) | Thời gian: 0.000s | Nhánh bị cắt: 8
    0   1   2
 0  X   O   X
 1  X   O   .
 2  O   .   .

Lượt của bạn (X). Nhập hàng cột (vd: 2 3): 2 1
    0   1   2
 0  X   O   X
 1  X   O   .
 2  O   X   .

A

In [ ]:
import math

class GameEngine:
    def __init__(self, mode):
        self.mode = mode
        if mode == '1':
            self.r, self.c = 8, 8
            self.board = [['.' for _ in range(8)] for _ in range(8)]
            self.board[0] = ['R','N','B','Q','K','B','N','R']
            self.board[1] = ['P']*8
            self.board[6] = ['p']*8
            self.board[7] = ['r','n','b','q','k','b','n','r']
        elif mode == '2':
            self.r, self.c = 10, 9
            self.board = [['.' for _ in range(9)] for _ in range(10)]
            self.board[0] = ['R','N','B','A','K','A','B','N','R']
            self.board[3] = ['C','.','.','.','.','.','.','.','C']
            self.board[9] = ['r','n','b','a','k','a','b','n','r']
        else:
            self.r, self.c = 9, 9
            self.board = [['.' for _ in range(9)] for _ in range(9)]

    def draw(self):
        print("\n   " + " ".join(map(str, range(self.c))))
        for i, row in enumerate(self.board):
            print(f"{i:2} " + " ".join(row))

    def evaluate(self):
        vals = {'K':1000, 'Q':90, 'R':50, 'B':30, 'N':30, 'P':10, 'A':20,
                'k':-1000, 'q':-90, 'r':-50, 'b':-30, 'n':-30, 'p':-10, 'a':-20}
        s = 0
        for r in range(self.r):
            for c in range(self.c):
                s += vals.get(self.board[r][c], 0)
        return s

    def get_moves(self, is_ai):
        m = []
        for r in range(self.r):
            for c in range(self.c):
                p = self.board[r][c]
                if p == '.': continue
                if (is_ai and p.isupper()) or (not is_ai and p.islower()):
                    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                        nr, nc = r+dr, c+dc
                        if 0 <= nr < self.r and 0 <= nc < self.c:
                            target = self.board[nr][nc]
                            if target == '.' or (is_ai != target.isupper()):
                                m.append(((r, c), (nr, nc)))
        return m

    def move(self, src, dst):
        old = self.board[dst[0]][dst[1]]
        self.board[dst[0]][dst[1]] = self.board[src[0]][src[1]]
        self.board[src[0]][src[1]] = '.'
        return old

    def undo(self, src, dst, old):
        self.board[src[0]][src[1]] = self.board[dst[0]][dst[1]]
        self.board[dst[0]][dst[1]] = old

def alpha_beta(engine, depth, a, b, is_max):
    if depth == 0: return engine.evaluate()
    moves = engine.get_moves(is_max)
    if not moves: return engine.evaluate()
    if is_max:
        v = -math.inf
        for s, d in moves:
            old = engine.move(s, d)
            v = max(v, alpha_beta(engine, depth-1, a, b, False))
            engine.undo(s, d, old)
            a = max(a, v)
            if b <= a: break
        return v
    else:
        v = math.inf
        for s, d in moves:
            old = engine.move(s, d)
            v = min(v, alpha_beta(engine, depth-1, a, b, True))
            engine.undo(s, d, old)
            b = min(b, v)
            if b <= a: break
        return v

def main():
    print("--- HE THONG AI ALPHA-BETA ---")
    print("1. Co Vua (8x8)\n2. Co Tuong (10x9)\n3. Co Vay (9x9)")
    choice = input("Chon: ")
    game = GameEngine(choice)
    while True:
        game.draw()
        try:
            raw = input("\nLuot ban (Nhap: hang1 cot1 hang2 cot2): ").split()
            s = (int(raw[0]), int(raw[1]))
            d = (int(raw[2]), int(raw[3]))
            game.move(s, d)
        except:
            print("Toa do khong hop le!")
            continue

        print("\nAI dang suy nghi...")
        best_v, best_m = -math.inf, None
        for s, d in game.get_moves(True):
            old = game.move(s, d)
            val = alpha_beta(game, 2, -math.inf, math.inf, False)
            game.undo(s, d, old)
            if val > best_v:
                best_v, best_m = val, (s, d)
        if best_m: game.move(best_m[0], best_m[1])

if __name__ == "__main__":
    main()

--- HE THONG AI ALPHA-BETA ---
1. Co Vua (8x8)
2. Co Tuong (10x9)
3. Co Vay (9x9)

   0 1 2 3 4 5 6 7 8
 0 R N B A K A B N R
 1 . . . . . . . . .
 2 . . . . . . . . .
 3 C . . . . . . . C
 4 . . . . . . . . .
 5 . . . . . . . . .
 6 . . . . . . . . .
 7 . . . . . . . . .
 8 . . . . . . . . .
 9 r n b a k a b n r
Toa do khong hop le!

   0 1 2 3 4 5 6 7 8
 0 R N B A K A B N R
 1 . . . . . . . . .
 2 . . . . . . . . .
 3 C . . . . . . . C
 4 . . . . . . . . .
 5 . . . . . . . . .
 6 . . . . . . . . .
 7 . . . . . . . . .
 8 . . . . . . . . .
 9 r n b a k a b n r


In [4]:
import ipywidgets as widgets
from IPython.display import display
import math


board = [' '] * 9
player = 'X'
ai = 'O'


def check_winner(b, p):
    win_conditions = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],
        [0, 3, 6], [1, 4, 7], [2, 5, 8],
        [0, 4, 8], [2, 4, 6]
    ]
    for condition in win_conditions:
        if b[condition[0]] == b[condition[1]] == b[condition[2]] == p:
            return True
    return False

def check_draw(b):
    return ' ' not in b

def minimax(b, depth, is_maximizing, alpha, beta):
    if check_winner(b, ai): return 10 - depth
    if check_winner(b, player): return depth - 10
    if check_draw(b): return 0

    if is_maximizing:
        max_eval = -math.inf
        for i in range(9):
            if b[i] == ' ':
                b[i] = ai
                eval = minimax(b, depth + 1, False, alpha, beta)
                b[i] = ' '
                max_eval = max(max_eval, eval)
                alpha = max(alpha, eval)
                if beta <= alpha: break
        return max_eval
    else:
        min_eval = math.inf
        for i in range(9):
            if b[i] == ' ':
                b[i] = player
                eval = minimax(b, depth + 1, True, alpha, beta)
                b[i] = ' '
                min_eval = min(min_eval, eval)
                beta = min(beta, eval)
                if beta <= alpha: break
        return min_eval

def on_button_click(b):
    idx = buttons.index(b)

    if board[idx] == ' ' and not check_winner(board, player) and not check_winner(board, ai):
        board[idx] = player
        b.description = player
        b.style.button_color = '#A9CCE3'

        if check_winner(board, player):
            status_label.value = "🎉 Kết thúc: Chúc mừng! Bạn đã thắng!"
            return
        elif check_draw(board):
            status_label.value = "🤝 Kết thúc: Trò chơi hòa!"
            return

        status_label.value = "🤖 AI đang suy nghĩ..."


        best_score = -math.inf
        best_move = None
        for i in range(9):
            if board[i] == ' ':
                board[i] = ai
                score = minimax(board, 0, False, -math.inf, math.inf)
                board[i] = ' '
                if score > best_score:
                    best_score = score
                    best_move = i

        if best_move is not None:
            board[best_move] = ai
            buttons[best_move].description = ai
            buttons[best_move].style.button_color = '#F5B7B1'

            if check_winner(board, ai):
                status_label.value = "💀 Kết thúc: AI đã thắng! Hãy thử lại."
            elif check_draw(board):
                status_label.value = "🤝 Kết thúc: Trò chơi hòa!"
            else:
                status_label.value = "👉 Lượt của bạn (X)"

def reset_game(b):
    global board
    board = [' '] * 9
    for btn in buttons:
        btn.description = ' '
        btn.style.button_color = None
    status_label.value = "👉 Lượt của bạn (X)"

buttons = [widgets.Button(description=' ', layout=widgets.Layout(width='80px', height='80px')) for _ in range(9)]
for btn in buttons:
    btn.style.font_weight = 'bold'
    btn.on_click(on_button_click)

grid = widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat(3, 85px)"))


status_label = widgets.Label(value="👉 Lượt của bạn (X)", style={'font_weight': 'bold'})
reset_btn = widgets.Button(description="Chơi lại", layout=widgets.Layout(width='255px', margin='10px 0 0 0'))
reset_btn.style.button_color = '#D5F5E3'
reset_btn.on_click(reset_game)

ui = widgets.VBox([status_label, grid, reset_btn])
display(ui)